# Deflated Sharpe Ratio (DSR)

The more trials you run, and the more their Sharpes vary, the higher the luck-only maximum, and so the higher the bar your best strategy must clear. Deflating the sharpe ration through this methodology answers the following question: is the best Sharpe higher than the best you’d expect from luck alone, given how many you tried?

SR₀ = √V · [ (1 − γ) · Z⁻¹(1 − 1/N) + γ · Z⁻¹(1 − 1/(N·e)) ]

In [12]:
# imports

import pandas as pd
import numpy as np
from scipy.stats import norm
from vportfolio import metrics, load_data, trials
from vportfolio.params import TRIALS_LOG

In [2]:
# Expected maximum Sharpe ratio from pure luck, given N trials
# returns the annual Sharpe that the best of N trials would reach by chance alone (the DSR threshold)
def expected_max_sharpe(sharpes):
    N = len(sharpes)
    V = sharpes.var() # more spread, more likely to stand out by chance
    # values under which does a share of the normald istribution lie
    z_1 = norm.ppf(1-1/N)
    z_2 = norm.ppf(1-1/(N*np.e))
    y = 0.5772 # Euler–Mascheroni constant
    SR = np.sqrt(V) * ((1 - y) * z_1 + y * z_2)
    return SR

In [3]:
# Deflated Sharpe Ratio: PSR against the best Sharpe expected from luck, given all trials
# returns the DSR
def dsr(nav, rf_daily, sharpes):
    sr0 = expected_max_sharpe(sharpes)
    return metrics.psr(nav, rf_daily, sr_threshold=sr0)

In [5]:
# function testing

# helper function to create fake sharpe ratios
def fake_sharpes(n, mean, std, seed=0):
    draws = np.random.default_rng(seed).normal(size=n)
    return pd.Series((draws - draws.mean()) / draws.std() * std + mean) # exactly this mean and std

rng = np.random.default_rng(0)
dates = pd.bdate_range("2020-01-01", periods=2520) # 10 years of business days
test_rf = pd.Series(0.0, index=dates) # risk-free rate of 0
# a strong NAV: exactly mean 0.0004 and std 0.01 per day (annual Sharpe = 0.63)
draws = rng.normal(size=2520)
strong = pd.Series((draws - draws.mean()) / draws.std() * 0.01 + 0.0004, index=dates)
nav_strong = (1 + strong).cumprod()
# 1. always true: the DSR is a probability between 0 and 1
test = dsr(nav_strong, test_rf, fake_sharpes(7, 0.5, 0.2))
assert 0 <= test <= 1
# 2. known answer: all trials with the same Sharpe → SR0 = 0 → DSR equals the PSR against 0
equal_sharpes = pd.Series([0.5] * 7)
assert np.isclose(dsr(nav_strong, test_rf, equal_sharpes), metrics.psr(nav_strong, test_rf), atol=1e-9)
# 3. inputs matter: more trials, or more spread among them, raise the bar → lower DSR
assert dsr(nav_strong, test_rf, fake_sharpes(50, 0.5, 0.2)) < test   # 50 trials vs 7
assert dsr(nav_strong, test_rf, fake_sharpes(7, 0.5, 0.4)) < test    # spread 0.4 vs 0.2

print("dsr passed all tests")

dsr passed all tests


Now that we have a DSR function, let's run on all sharpes, benchmarks and diagnostics both. 

The book’s effective number of independent trials, which would be lower given the candidates’ high correlations, is not used: the plain count is simpler and stricter.

In [13]:
# import the data

log = pd.read_csv(TRIALS_LOG).set_index("name")
sharpes = log["sharpe"]
navs = trials.load_navs(log.index.tolist())
shared_data = load_data._load_shared()
rf_daily = shared_data["rf_daily"]

In [18]:
# run the DSR on each strategy

threshold = expected_max_sharpe(log["sharpe"])
print(f"Expected maximum Sharpe from luck ({len(log)} trials): {threshold:.3f}")

results = {} # empty dictionary to populate with the individual dictionaries
for name in navs.columns: # for each strategy: PSR against 0 and DSR
    nav = navs[name].dropna()
    results[name] = {
        "psr_vs_0": metrics.psr(nav, rf_daily, sr_threshold=0),# probability the Sharpe is above 0
        "dsr": dsr(nav, rf_daily, log["sharpe"]), # same, against the luck-only bar
    }
dsr_table = pd.DataFrame(results).T
dsr_table.round(2)

Expected maximum Sharpe from luck (7 trials): 0.104


,psr_vs_0,dsr
sixty_forty,1.00,0.99
equal_weights,0.98,0.94
risk_weighted,0.99,0.98
hrp,0.99,0.96
nco,0.97,0.93
min_var_raw,0.96,0.91
min_var_denoised,0.97,0.93


In [20]:
print(dsr_table.round(2).to_markdown())

|                  |   psr_vs_0 |   dsr |
|:-----------------|-----------:|------:|
| sixty_forty      |       1    |  0.99 |
| equal_weights    |       0.98 |  0.94 |
| risk_weighted    |       0.99 |  0.98 |
| hrp              |       0.99 |  0.96 |
| nco              |       0.97 |  0.93 |
| min_var_raw      |       0.96 |  0.91 |
| min_var_denoised |       0.97 |  0.93 |


The luck-only bar is 0.104: by chance alone, the best of the 7 trials would be expected to reach an annual Sharpe of about 0.10.

The DSR is so low because it is based on how spread the sharpes are. In our trials, every strategy has a relatively comparable Sharpe ratio so the correlation makes the estimate rough

**Conclusions**
- Every strategy clears the luck bar with a solid margin
- HRP is the best candidate, with a DSR of 0.96. That is just above the pre-registered treshold of 0.95. The candidates earn a genuine positive risk-adjusted return, but not a demonstrably better one than equal weight or 60/40
- NCO and min_var allocators are close to the bar but below it (0.91 and 0.93)